# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdPYty47iRv4J4rjLSRpYf88iectqUx9bMOOtXLHkmV5aPRVGQxZgiFT7s0Tr+9+tuACRAgLI8u0ldHXfLkkg8Gt2NfoPzuBUk8Sy87S5XWz22FS6WSZqzJBvH8mvKy69FEU7LH3m4qJ5k8yIPo+rZPOX+NIxvx/EsTRYsXy3hB5NPz5d5mMR+1GEnYZZ32FEYwN+DeCVbL/18HoUT1fwCfo7jcfyKHfGcBzkbxPdhmsQLHuc99ilJbiPODpPIn7D7jJ0kgQ+AHA+9w/OTgw+sD2vp4ohd/g1my1rjrR1YcQ6dx1tt5sdTFie53eiwN8Zrq41TT/kM8JAl0T33pslDHCX+lKfeNExbbbb9E8vytDeOGVzjLfqPvh+tYn8RAjjRikUAVs4zBe5RGt5zVg3FYChYWpKuOiwrlrhuRFg2hy9BkWcdMWJX3djO/fSW59n2ZLUdTgVsBWK1QyvK5n7Kp2yWRDB2xh7mIcx5W/ipDwvnOPIM7mRiUGjKzgZfBpfsIQ1zQIwEBqDOExPgNEnyrr3QcEY4VDiXqMBr4mcc0aSRwZ9k+NlSv+EpoIm3PA9h8rx2u+qe8rxI47Lr35MwbqkhOwCCRoss92+R4QTBsPMrttdlF2m48NMVC+Y8uOs1IrBlLFM1YrIRS2LBYBI0edtDdMDKKo7amWL/nYZZNHTV2E0fsK2hL09X2i+8ZkkKO2/KQsJKBL2RCZv6l+N4OJtGBMKk3quDo7btnpK0ql+Y0XQ0nGsivBAXYVxwBxjWctQFuEtD2B59ds27UfLAcWPhWnl9pWLqG/cwAK0fr1p32EkNiaPQjWvgFz/3x1vIOan/IL/wWRhzoM1Nm0FL7M67PJ5mDyEw6XirGy5X8QRFRQmOHLlp/XhJxhXQuptlxcQiCDU3GZsYumGtNS6CEYVAM6mFt58HFVrZbfi3gC9B4NIHSJeNKf5Mx6WfZdU23e+yQ9yeLPDjJEaBid+mIVCLl4IxVOKqfETcUo1r7cLTFe3mHR2XnRe1Z9tsKHfy+q5Dkrf0PTPmE30ktyIDBchA1RI0vNj0DOpkc+8fULWRxkn4kwRs4N7O1FznEjkb3nexD91vILwGsxLrYhiUGbjl6k+bkA5c3rGGqjeGRjfPMzICYDbagIsFQzYsTeDBooY2Z1Ax85suG3I/DebMvVLiApAyUjmD6eALXY/yhr2umOe1GHGxou4uNVPirkmtyL7PaxSUaKuapHV0LrsUUeQt69JLduiI0dzMR4+UgO/S2pWgNUReA5mbFYgghnsjCHAbxKgE7LfcFdqw/3d2R4Wkhl2irg1lPl72rnEsXeBL0uD57SsafqcmeVtuPiGRhT0nYcw0Ie3aTroQb95T+ijft7GaRij7OXeX3us32WIOpl5PpAYCqQsMCBtq0XgDi8ZpzQAsbnvmWSCpzXcy0RGf+UWUsxk4bhPfMEsqa056g9M1ctlhCGi9bMkgHrVLxisb/ytEhbNRTYS8fI1qlVLEiO54++j869nJ+cHR4NI7Or6EAZu86XEMK685kifnhwcn3tfzy5+HFweHAxOcnC+WODWPMi7bfzgYDuQ0L3U67akMdlYjI0OXQQT8IaBoizDFME+LAFAAmDPcyiPdlL08+FqDkWYwEVV6KjDy5eDj8dngaMNOyqvROp6efzkeDF39taGx8yK5RxD1vqMvG/TL772Mp/Wu56PPJdXX9U7yuZQNRwejg82WKfw56HI6GB1Qt78Mz8+sfnI8WhvPfezV/XuWxCXFBss5X/AUdjiFj9iXU5NYgi0aSFbjGUkzLxTDy6601k17I/FULGN0eTD87A1HB5+Ozz5tOECe+tnciIiIRk2IdQxRYlY8W4tfc3A3lkeD0ws5dW02oPfl8ReFG5PGMuwH+iANJ0WOG+riih1GRZaD6fx7cBuL+C6DDUcrNWm2BmU2I+m4Ovx8dfbzcB3KtUckCQgKDVtyhM0wrfU+PLkajtybRcduINZvdMEhB5fOObVRsfdDkt7x1Jzw9IA+Ph6fWCKv1nvh49QVXeWCmwG3mUMH3+y+ZhHWPPWlmA3WLMg5Un1ZdbZTLHdIsfIi9dGKkKziHV1dHoyOz8+84eAQZtt/u9vdLW0LMErZIowLjFcsYYQ7vpqloHpYxm8xgs1aCZgkCxA6Ez/y44CzZMaEAZInEcgkvPV7Bvac1CIfr4YDdrxzrjjVOxmgVhodnw7Or0YShv9UILyCrzAVKKtpxiLuZ5xNS/AFvr3Pg4PL0YfBwag2yNt3MMgrduqnd0ygGmCbRaBYRFiQzcHWzifcz8nWffsuk4iDnY9RWorcZz1TCaoQPY2ImzZIlislA/56NbgCSfBhHff/o+AF704nGtWbutm8Z3RGWI9lZLfD/OjBX2WsALgjAnA4PGLDv56EOadoOQbDeRQCLTEsZayJSIJttqEnLWrmhxEYANmfGKm1hxBGxZHNZbqMHW0t5uIME0drVR+S9Fmcwews5gHPMgx/a1E0Rvb1OEaSCdtSZitqQlFqO/ii62nbmJBxMdNSqBqW6r/DKkIYAtSWt3BP354O8dBxmmsdZijq8mepfDvMUqeqUQVcXYaXd3SYGgXXOFYmtOkRAmMu/DsOhMha044ggpfc9UdpwaXxucZLUR7KK5RAQZGmPA5W4Osu/OU8AT4DCfFV7NAT4NAc2p4e/M1TJFXwAbe8EU9Ozr96w4sBUtVus4tiYxdYYxqKpNXV2cnx6fFocISRXjF7jhJpO1tyEI7KhpezSnRX4+1RCk02AhNRAg1Pyjxht1xKywU4IAim82g6zznSWQJSCcDGzx4rYtyoKLgDDV3I8RmCDXIWtnMaBncR16GHxvegTwD1G0FpLlQKlFG44EmRk986JPR89fNgPk1u2WVBObdyccCFl3WZi4Jba1LRafT5cjD8fH5y5H24IKTu7rIf4O/+2y7RC3///GEnc3c25tjbp0kMRAtYLgejS9jWqMRwzOM4zEOQhH+An+DUyQzNG5CEOUbLcvR68kwJ0u0FXyQga0Z+dsdGqS8lISAdg03bmC9mEYrLBXjTKIswPZxDYwrk38jvHopQA+mgLu7QMcTHXpAUMSpi4FIx76comUhtArfRKUuTKNP4TTzx/CAXYVTccONYOEgbPWwESOSFMx+x9AsnT5KcSvzTw5ywMzc8lO0zpjqIxECaZNn2MvJz+LWAhSxAj4WTMArzFTGTmaH0ZzxfdR1JWJrcyqCOt+7DKU9UW5oVnfBuVkxa6etrTHXv/PDn3njrv376583rDsHaoXZtd5fxOPsDtWNlwy7aS8tW24gC4JPrHrDcDUIICyb0tNlPyIYMVRo1UdicrFDUecKDmfKI57xFLjpqdQ82JoUfKuxOkiRypN4/0DD1fPsIR2UfQiVZTxMMBYKCFgqeig+MDmEMSh+fR8ot5KWniNyRLf0AO6dZLjUSNI7RzluAiIkxhZ7CpsBZwlwMEq26bDQPwRhM+T00yQzY/lHA1kIuzooF6YDmbLuNFcxfOmoZ7IZtm0E++kCJ5/SWQ3m69BheRRyCreVJrpkpn/QRS0i6+Odtq92d82/XvR9vnrxHBTKajbSHHEA/KQTgRbgEBsnyutHnAFIDRgPxFdEfTVqT6AkSXRCcqiRWYPkvqm6i1KWLVHVA2dEgM+a6ApUU3wkeE4NPw+xuXfpPG6gWr4SGgq1azskkSYUscxkWzAcrULcuQBbnLSDSNXGh2Dw3oLhSDEP0wOYoIlEmI7YnI2KS2Hq0UfDUY4/8yYjy2sFxe7nr2dReuqN9Y36uQoULHc5ocG0EuT+EkArAkYqLpZiVyic8kEf+ilQAmHfl7V5Z63QNDW6kndJ2CKxh4MeZXWzDWu64q6hGoEkZuDO488l5Nep6ggBkPqhNrD9SpT3ojXAgY1XjUtUOsTDLChBVMBh5NLfzBLYX+Geg96V4o+Ii0D2aYIMRF+DboK53CG+3/KowBHMK200VYjmKiAQJ6ulQbQwAeIPkqEM0OnKcajKXLKyqVBx50mdteHPkV+gdgyUTZxopp8pzRmyoIJpOU8FmS9XTKIAA1YxWAzRupeOt/xGu7g/jMfBH94f/QC0NLY4/nZ1fDg4PhoN2p7FrGb7D3shW6/rfaJkThBb0WsbTewoxsAmfoQ0tN0xPNTyelc4g2N+AUaTN3Ae6KgcdaWpEEXH2LCg4qeMJyv1KZyMTuOiFctGbWmU2jQEB1c/NLHK0ulSi7W9PUmbDG+aQ89TTzWIwM+2FRVpgMFYPf2K7btFYaupqiU0KWl1SnWEUZkb0F5N0FPIcXSp1cSjoesMuiTZTdu9H4bQiIlFHk2agWB/luE9apV5JK6T4y6L3G5AMOzmJ5ppNI1vzZGtoR5M1U088/pfTD6fpVCj9Hhoaq28mJDardP3mhkY5qzQyRIFm5TDSVifBV1kTYhySJZSU3BU3jDy7FNHanCIF7Sh/bKxXsWoeCbHlOHVesqvfECJoifCYAts2guAJUD0P5i0Bhas2yUUqzNFhgTCpDSIJlhHNwJFFfOFQTxbL4uX0r8qV2e0Vtv9AYRxrNHCJ70oFq9qa7O0A/1A2LJbsUXZ6UhpwslPZMQ6PTFQgq+jtZRGLxUsFg4W6i2RaQHNZPI559CImDgtlqNcVdX3GooPpQP1tzXkErp2MCzQWzIPD4C6SJznirKWvlc8XaRSFEzB0QZBiqNa8zdM0SauuxWSZJhjn3bD6vqy279QK8ccVcOJIgDJ3QYJ5t8vCC+NZIurdcQy0aGmcG8OcPY6zJQ/ArRV+E2j1dPqAJea4Kc6+HB8dH1ASD1tlZgADhoc9+lixzngLp6XwAKB6vHUI/USYs1aOeZ/6C28xwVa7xgOiZyp6q3menG5ugY5khcwuySMPbi+LvGVy/vV4K74Pp6G/nS1Ckf/f3gZSpattgLeP8HZELKxLkbKOgMKTsUXVA+M8ft4PsvtOnMw51kDAF/BV82y8ddMx58z5t7xvujG1eAvtNSz2QatwqZ4JWUQVSEXezZZRCKsB5G3pJdQyMEO9QTv12Zu6oALSXOvUQFeGml/v3jiblhTBligAROu9m7oaFc0Vnapx9282DIrjN+mk4WCKacPMi+95HHj+vR/CrgcJ7g4YUelxhjg4+zI4O2Rzfh/szPffvwW9ECR0fgLZV57KAKmVoBcFbosUAzoLv4SnYNWz2WLJbyU/qNmQ9hW5OyAX4W7a14Y5Gnw5uzo5sT3+8RYCL9Y93pJEF94RLqj2YEOHpXJ9gwh9/88kBMU2dDiyB0WexMkiKTLa5pcUSk1VdiKGJWoCOQMHlfny8I6Qrqh6ORDpTnqOR2EWJLhz2KkPcN+HvjrdciS0N0AVLjLUJWCU7OsZCgp5MpFfl9tJIDmjYUrJRHQXB2fEkjMKv6JunKo0bWYcitEhcHi4GFk/PD87vLq8HJyNZCoJg+rKhkEG9UCehrnntTIezTqSm7xwuj5mgBd26CqBDM9N+axHqoBh8Qlyod7H3MqgZ5aRH3CQC0I+C4YEEaE9GfEs8qsHSroQb50BSvXYHM1Vrgcmr75D+5nK23uPJYCqXPGpIUBojY58pAUXBU+y1qO5zNdqla9vntrWGCLn74FWdaO81lxybY8kSJkqMJqINIKXRUkOZispSRB9lZ69NvXmDc70uNuj2TpsT3x5WjNocy7C6CKigpidqZ29IrsX2AXM0bQllDxStF6RQ2SuVdi1OyReSgPIsC5118UBge3FVNtgKve3V5GDhiBhbZBF2wFis10CDjIm+rEJ9wNpNwqzUe5R3Xo0ZDXJa3m+y4ij4GXjyFFZgngyMCtPUjqLXewqk3bnuTkba1qaZl5TbOOYv5pe0+F0wAw1hMTNs3Fb94GzxkJ9OuaQLMHcyKnWDSEjqQzc3B9vFfls+0eqTgbrf021OAXG+mRSdzHN15o11PcDP2GwED1ZWE5LIQH5bN3RKuxGwU4gCHyvFVLP83zZfEoBL6k7oWs3FcJS1NfaPTast3dGpYWYqjaTH6MBGXAvT4QrI5ULAJ8XGWXNqHo2nEbchF/sjK9piPn+IAFHA03V7VnKucqt6jvMoQzXbjRJrkdzbaUmwLORvZre6NTbljK/bFvesdrqfsMa1Wf10xyJerfKorV6CfRSJ/pmNQDDKffKailsiI5hF/+06mIA6z9vQ+RRPvWsxlXbp18lxvSKlZeKsars7qUibINZnxFh2tzN4guVXqMEc4smXYPlTpVV7/FAsQHrCCf11W0cg6mf3MFEGq8UjGJkWu13SEcSidNisWzhpgNYOljJA4q8v1+bdeNzcZrRipLPk4EJKVx4PF3C0nMSLx3a6r1Gk0eZs4Z2r7WxxNIQz+OCcGOfR6MLdnE+HKEh+WkwYhISgbxJmjxkIKiEL52hbPLvkxCrN5NiOovQ1p9gElNW6VlSSgb+6tZhLXSsi151U2gZIPtjrfPTo0KPYcTaTAgr8aSUNIfGS63IkqACR1ew6u2DWzqNgJGO0+SXMIr8HazfbH0NsRYmY2cjtrfb3f0Tgxvv3/6JfXv/ts0OluDmfOWTn8N8592bP3bfvGetnz+PTk86LArvOPsEDmvSZodzsKj4DhUO4X9s6M/8NJRdjGBMBdVBgHwlIPJhHiwmA3rviB3g7PK3beEnbh8f2QrB7PBk/sSkImIvFLkkKgOzZ9CQXG6TjPZJuytcw1a1yezukgygDA6Fbbw9Wi1lNMS1RovC0M6M7XUv5VaC22Lr9BWMHTVdX37WACKWr40GP0mCwO8OaQ1w8/t770hepDxbOlAiDhphvAg1F+uDl7rryk7gNUmmK3F2Z4mnK6fgoU35s1gTa6dNU5prWQvH0hpLWaSHOLu42wf4rZ5GkHCjagDf38PCsPGWOlVZiqN1kXQ9iHED3JKHAe8J8fKIudApf8I4T7V5MaqOa4YF2IF1p0hwrejq8uTfuiBg0xjstDK3/xutqUljOPs0mKrKyPGQj6XzVzl+WpQOL+XviS4q9FZZobQXyvgQkfHg4pgMeJE2+EC2qyXwAQhv6a+QI//fWKdOC7FklHEx/fHNFP4GfI8phJbHzwmFQZKkU6xyT8qIiqbP2t1u12CWlAoiqJlpHYDD4y/DHWm+7SiCi0KCEu9m3g0HE3G4TLpryrZuo2jSbWPAgokfMFXCW5EZrLqrmxKnnRppakyvoWn/j7vv2LAIMOKKCbIVq6ZuQBZs6tePasKn178z0KQV75uz6TQJ9tgVVdLUXkdDDpbyv3CmIk+2VeRElIdmqzgALR2Hv5AGUlSqB5NsB5Hw2y99QmvjijhXtXEzkB+VLyNtwAVWBwfZZnafc3MLG4/qg5dpcgsUFJxQnTEBcw5Rbm3iagNLKNA4fHyyGlwbG/pGMa3DvtiUpTWPrsNsfl6Pcdm+xuadyju3SYGIaz27RZI7jfKiXE0pFUdk7Tnb+yKJIhl7R86L+beclSkVEVyXOw/D5BZxlLIR1rGBvQqsP5c06Nf9JVNaN5BFTfKcLJn7mdStVj1IeWT5WcWlUgMVQjvs78mEIvcU7SWcwM9v8jfYYJ64h26i9JKoehHrQkBo99gMOAFRBKb1mh1STPBoh8S5BANVe0zafoP98asUnFgjNhTfrAblurFN+cNqZqID25p3rA46qrC58dul7DbdvgYl3Vv4RRvPJ4dHnrquGEY4FmKFkl9wwl4ti97Bsym5YiNkhmZWqH4PvvGAzhaqbNY2QoHnBjF5JTKaeerHGcKA5l91tg1LhvCwIQHW0cVWjOEGrGIPk2kYVCI464j6dlESyvCUcjaHabqLu3sjJKgnxlxQ4/loUWohmLlPGLkWB6d15tCsGeQSZ586Q93o5Jt5wexWtpVUU8erPVnlAIR/fNJoLti7Gl8xvjZuyd1VK437b3S16y5MqyGgLo7cdn1vCAyy/SjZ5OmGkS/RY8OkSAMli8tySpoJKX4kTkw81uZsNPnLfK+5Wpltsyr160up+okkEDy33mXiShHBasOY0lzlCr3HamoTXNFZJNg3GhqZ5JmxzSDiDF2vVrmETjVlnViOUvaXpUS0mvZfmx+ontHbDw9xiyMB1GFju6CqOusANj7KBU0Q1EsLTUp3KgK3zZn3u+xDEUZT9vEjFjhoAghE7gJkqDYHjEBHp8ShLrlhlXzVHgrTKAZL3GCEIJ1Z3eCed+9H2OONHvO89xCGwGpPdQLimZjlc3f//Tu2wz4PvhyaVaaZR5UhKO3hU5RSqGEpF36APbS7VV80PqgAAwtiHMUpNaEhp7J5rRzHwQj3np/eijzA1nbQuxfL0Ws/qNSESrTlm0uWb+XN4B+iPWa8AIP1Vyw6/JfmGaNw8o3wI0ZOZ/rAFgjowyD3GSLWMd/3LV6riPl3Lv7d9yxe4+vFVIxr1AjFCZ45UBVkK/lJNWjlXjTruoz99bt+uYXqNayLKQivHDQ/1iVt3wuYtb563ZbR2BjmB4ERy4Tb9u2IL97F6LwX81ufShzyTKWp77j3C08TVx/lHEoChkve27PalbK6un1Tk1FvuuzEB8dsroSUrK4Svj35oZXrKU+sOuIp6xX0uNh/7++xgczgiPeVsFePpbHwhAbYX5IJ3BNmhgg7Npex1KMvCLVZZHZBEWCgEZWOYQBYf3h8MVhTUmYUnk2KGVrZ/T0DcyKVOfHoOG/fnZrEAixvGUiPpnZ/tsyc98VZ8j6VRlbvksLL/Wo18R6KmDDQFSt1bFFq1acPu1xSXVj/KBpoOX6As/+9byLUFkquXUsML6svcdjrvRsFT0PI/Fe/h49HznURnteszCDGOrg3nhFtL2QRr1iH0Th5aOIn/YIZWth02+BEqlrF9NYaLNGmqkWvHOk0/XrW67U7bOQF291gt1AHuXOebQ/MVbaH78+2J4KWPejX833KsobxlkpC15Na+vXUQDK8amIDKFj3eGAXP/jAZ3ZSqPE4SdnxLoyiOsNs6kOVg08LCoSrldqnWPFq8pHwKAdCIp6Trfs7fJOG+0h25Utgg/pJIe1pv28dFdpwXZbpja+kgT3dAhCwwH+KLmENZCyS3HC99K5Nci+qUlnpgDa8PF7z5LVai/oL+Wpefru5DqPunZRY69RDBaZvJyNHmuOo8F7vBh5AC1+wId+zYWjdTdU/ZhEcWr8MnbQeFUC97t7siZ1+aHfZEGwNUWiuqqlqxsu7LqX4ZisrCVGLPpvBSinCNLFUR1WnxE9txvddcWpHix+hf52k+HYh7YAqofhF7rNbk27oVEt2anCfX1jgovF7Pemh3gVCtI2SZKkK67SQnaOcTqQpRbWuSNwwTO/AlqAjMBEe8F7439g+fRcvCIMnbSuGK/4FB6MO2fKNni9dIXSjgs0izpetN67zXuVhthqV8KUuMqiq5RPsQ5EU2rTHJavaXc/cQFar8bVE9o0MutndnktxVUrMdZASOxuxWpykJHFffjZVbv12a9MTDr9uGvSW/XhVvd4G/3WDTK+REWfP6aBifeDuvR8VePbNbTCLE9v6+GvX+oK8o7o0Tt3tvqtvbbdHXuPuagPDrhHJL2uTkgSVBQWcoukhhcmTJRgt6CCiUWCcK8G9KzsITFkZF5nYhdWB7vjjvktzlElf/uOuPACD48qTDHikAl9riQeOetIfLJPWTzUPUI4I36XAOT5SffREWkMfnFX8wzrVVE7X0z5eIYsO8NnpB/bl8uC03TiNSEUMxSkqmOf1f/OMtT6rMzgHVdai/VrGv+zzW/S6oNdnCWvhUcCP8gXN7dfNq9Nel6YW5zyd8yTPCYF8XhRR7sc8KTIwYqREdoxvkLemdjdhd127EqVNJtMZrISicolJpagXZ6m78gVnVDUEdt+tKP92L9hSw8axkhF9k/9CTF+k0uoqsMMw1tNv4Y1OG6vXQAvHrnLZXPiDdW0hF9H1l0uMJeUGUp7VeBTgVqfDGB0an6U8c9RlXF2efKeyrLVQCtB1ZsUtJZ+fomIoJRQC+Dvb2y3XBpYgLemwWpJiZa0up7H0TqDQVW3l9OCdYrUCkkpj3szesK/gq6nj+hbGZakK/O84HIAmT3kOxSonskBfs5ceBAwewCDX1DiUphjqqkRdTabPK3ZhZULp3EPur5hQfWaHlylqEHS/qX52hzkanHkpaSUAJF6ViKp78xoG93bLt9/B3vTkqbAFcEFLiRUpvvrGcVHFcabJL8/RooY2XyPxM19NElAQx/jGv7RY6vG9khsBkFKTTveU/pSKMFQ9sXxqhW9gTcG7AseRWBdfEVhnPwnMGqaTr8CVMjzEg5yoID1PFAF4hAfPK0PsFobG8dbT/wJw8ZpB"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.

import time
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main()
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
